# Global Organ Donation & Transplantation — Comprehensive Case Study
**Case study deliverable:** data understanding, preprocessing and EDA, learned data insights, 15 professional analytical questions with Plotly visualizations, hypothesis testing, and four-model comparisons for regression, classification and clustering, followed by evidence-based recommendations and Tableau dashboard guidance.

**Dataset:** `final organ dataset(3).csv` (place this CSV in the same folder as this notebook).

> **Responsible interpretation:** This is observational data. Results do not establish causation or support clinical decisions. Check data provenance and definitions before policy use. `Avg_Wait_Years_US` is US-specific by its name and must not be interpreted as country-specific waiting time.

In [1]:
import warnings, os, math
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error,
    accuracy_score, precision_score, recall_score, f1_score, silhouette_score,
    confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.inspection import permutation_importance

DATA_CANDIDATES = [
    "final organ dataset(3).csv",
    "final organ dataset.csv",
    "final organ clean.csv",
    "final_organ_dataset.csv",
    "organ_donation_cleaned_tableau.csv",
    "organ_donation_cleaned_powerbi.csv",
]
DATA_PATH = next((candidate for candidate in DATA_CANDIDATES if os.path.exists(candidate)), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset CSV not found. Place one of these files beside the notebook: "
        + ", ".join(DATA_CANDIDATES)
    )
raw = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", raw.shape)
display(raw.head())

Loaded: final organ dataset.csv | shape: (2138, 18)


,Country,Continent,Population_M,Year,Organ,Deceased_Donors,Living_Donors,Total_Donors,Donor_Rate_PMP,Deceased_Rate_PMP,Transplants,Living_Donor_Pct,Deceased_Donor_Pct,Waiting_List,Avg_Wait_Years_US,Survival_Rate_1Y_Pct,Survival_Rate_5Y_Pct,Can_Living_Donate
0,Brazil,Americas,243.6,2026,Heart,436.0,0.0,436.0,1.79,1.79,482.0,0.0,100.0,2056.0,0.5,90.0,76.6,No
1,Pakistan,Americas,1.0,2016,Pancreas,2.0,86.0,88.0,88.00,2.00,0.0,0.7,100.0,1004.0,1.4,87.4,72.6,Rare (partial)
2,Iran,Europe,90.9,2026,Pancreas,39.0,NaN,162.0,1.78,0.43,149.0,4.0,99.4,180.0,1.4,89.2,70.4,Rare (partial)
3,Mexico,Americas,132.4,2016,Lung,76.0,35.0,111.0,0.84,0.57,87.0,3.3,99.4,200.0,0.3,83.5,59.7,Rare (lobar)
4,India,Asia,1350.0,2020,Pancreas,21.0,348.0,369.0,0.27,0.02,359.0,1.6,100.0,4493.0,1.4,86.3,73.3,Rare (partial)


## 2. Preprocessing and exploratory data analysis (EDA)

This section documents the original data before changing it, validates data types and ranges, handles missing values transparently, standardizes category labels, and creates a separate analysis-ready table. Suspected outliers are flagged rather than automatically deleted because unusual donor rates may be legitimate.

In [2]:
# Preserve the untouched source and create a working copy.
source_df = raw.copy(deep=True)
eda_df = raw.copy(deep=True)

# Convert expected numeric columns safely.
numeric_cols = [
    "Population_M", "Year", "Deceased_Donors", "Living_Donors", "Total_Donors",
    "Donor_Rate_PMP", "Deceased_Rate_PMP", "Transplants", "Living_Donor_Pct",
    "Deceased_Donor_Pct", "Waiting_List", "Avg_Wait_Years_US",
    "Survival_Rate_1Y_Pct", "Survival_Rate_5Y_Pct"
]
for col in numeric_cols:
    if col in eda_df.columns:
        eda_df[col] = pd.to_numeric(eda_df[col], errors="coerce")

# Trim category whitespace; normalize continent capitalization without overwriting raw.
for col in ["Country", "Continent", "Organ", "Can_Living_Donate"]:
    if col in eda_df.columns:
        eda_df[col] = eda_df[col].astype("string").str.strip()
if "Continent" in eda_df.columns:
    eda_df["Continent"] = eda_df["Continent"].str.title()

print("Original shape:", source_df.shape)
print("Exact duplicate rows:", int(eda_df.duplicated().sum()))
print("Duplicate Country-Year-Organ rows:", int(eda_df.duplicated(["Country","Year","Organ"], keep=False).sum()))
print("Missing values by column:")
display(eda_df.isna().sum().sort_values(ascending=False).rename("Missing").to_frame())
display(eda_df.describe(include="all").T)

# Flag impossible/unsupported values for review; do not silently delete.
quality_flags = pd.DataFrame({
    "Check": [
        "Year before 2010 or after 2025",
        "Negative donor/transplant/waiting-list counts",
        "Living donor percentage outside 0–100",
        "1-year survival outside 0–100",
        "5-year survival outside 0–100"
    ],
    "Rows flagged": [
        int((~eda_df["Year"].between(2010,2025)).sum()),
        int((eda_df[["Deceased_Donors","Living_Donors","Total_Donors","Transplants","Waiting_List"]] < 0).any(axis=1).sum()),
        int((~eda_df["Living_Donor_Pct"].between(0,100)).sum()),
        int((~eda_df["Survival_Rate_1Y_Pct"].between(0,100)).sum()),
        int((~eda_df["Survival_Rate_5Y_Pct"].between(0,100)).sum())
    ]
})
display(quality_flags)

# IQR outlier flags are for investigation only.
rate_col = eda_df["Donor_Rate_PMP"]
q1, q3 = rate_col.quantile([.25,.75])
iqr = q3-q1
eda_df["Donor_Rate_IQR_Outlier"] = rate_col.notna() & ((rate_col < q1-1.5*iqr) | (rate_col > q3+1.5*iqr))

# Deduplicate only exact duplicate rows after reporting them. Keep duplicate keys for investigation.
analysis_df = eda_df.drop_duplicates().copy()
print("Analysis-ready shape after exact duplicate removal:", analysis_df.shape)
print("Donor-rate IQR outliers flagged (not removed):", int(analysis_df["Donor_Rate_IQR_Outlier"].sum()))


Original shape: (2138, 18)
Exact duplicate rows: 6
Duplicate Country-Year-Organ rows: 572
Missing values by column:


,Missing
Deceased_Rate_PMP,74
Waiting_List,74
Population_M,74
Survival_Rate_5Y_Pct,74
Survival_Rate_1Y_Pct,74
Deceased_Donors,74
Living_Donors,74
Total_Donors,74
Donor_Rate_PMP,74
Avg_Wait_Years_US,74


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Country,2066,44,Turkey,92,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Continent,2138,5,Europe,938,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Population_M,2064.0,NaN,NaN,NaN,82.401841,215.547382,1.0,7.8,35.7,70.1,1532.7
Year,2138.0,NaN,NaN,NaN,2023.324135,16.361104,1978.0,2019.0,2023.0,2025.0,2218.0
Organ,2138,6,Heart,359,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Deceased_Donors,2064.0,NaN,NaN,NaN,110.721415,431.059247,0.0,4.0,18.0,67.25,7196.0
Living_Donors,2064.0,NaN,NaN,NaN,130.145833,385.220666,0.0,2.0,14.0,81.0,6056.0
Total_Donors,2064.0,NaN,NaN,NaN,239.760174,726.40069,0.0,12.0,50.0,178.25,11098.0
Donor_Rate_PMP,2064.0,NaN,NaN,NaN,23.394496,196.094316,0.0,0.66,2.065,5.9525,5405.0
Deceased_Rate_PMP,2064.0,NaN,NaN,NaN,7.984855,60.771292,0.0,0.16,0.9,3.0225,1785.0


,Check,Rows flagged
0,Year before 2010 or after 2025,285
1,Negative donor/transplant/waiting-list counts,0
2,Living donor percentage outside 0–100,74
3,1-year survival outside 0–100,74
4,5-year survival outside 0–100,74


Analysis-ready shape after exact duplicate removal: (2132, 19)
Donor-rate IQR outliers flagged (not removed): 244


### EDA visual overview

In [3]:
# Missingness chart
missing_counts = analysis_df.drop(columns=["Donor_Rate_IQR_Outlier"], errors="ignore").isna().sum()
fig = px.bar(x=missing_counts.index, y=missing_counts.values,
             labels={"x":"Field","y":"Missing values"}, title="EDA — Missing Values by Field")
fig.update_layout(xaxis_tickangle=-45); fig.show()

# Distributions of important numeric measures
eda_numeric = [c for c in ["Population_M","Total_Donors","Donor_Rate_PMP","Transplants","Waiting_List",
                           "Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct"] if c in analysis_df]
for col in eda_numeric:
    fig = px.histogram(analysis_df, x=col, marginal="box", nbins=30, title=f"EDA — Distribution of {col}")
    fig.show()

# Categorical coverage
for col in ["Continent","Organ","Can_Living_Donate"]:
    counts = analysis_df[col].value_counts(dropna=False).rename_axis(col).reset_index(name="Records")
    fig = px.bar(counts, x=col, y="Records", title=f"EDA — Record Coverage by {col}")
    fig.update_layout(xaxis_tickangle=-30); fig.show()

# Year coverage; annotate that source contains some years outside target period.
fig = px.histogram(analysis_df, x="Year", color="Continent", title="EDA — Dataset Coverage by Year and Continent")
fig.show()


## 3. What the EDA tells us about the data

Use the output below as the evidence-based data-understanding section of the report. These are dataset diagnostics, not policy conclusions. Revisit them after running all cells.

In [4]:
insights = []
insights.append(f"The dataset contains {len(source_df):,} rows and {source_df.shape[1]} columns before cleaning.")
insights.append(f"It covers {analysis_df['Country'].nunique(dropna=True)} distinct countries, {analysis_df['Continent'].nunique(dropna=True)} continents, {analysis_df['Organ'].nunique(dropna=True)} organ categories and {analysis_df['Year'].nunique(dropna=True)} distinct years.")
insights.append(f"There are {int(source_df.isna().sum().sum()):,} missing cells across all fields; missingness should be handled per analysis rather than filling every column indiscriminately.")
insights.append(f"There are {int(source_df.duplicated().sum()):,} exact duplicate rows. Duplicate Country-Year-Organ keys are separately flagged because they may reflect repeated observations or data issues.")
insights.append(f"{int((~eda_df['Year'].between(2010,2025)).sum()):,} records have years outside 2010–2025; keep them in the quality audit and exclude only when an analysis explicitly uses the 2010–2025 window.")
insights.append(f"{int(analysis_df['Donor_Rate_IQR_Outlier'].sum()):,} donor-rate records are IQR outliers. They are flagged for validation, not automatically removed.")
insights.append("The field Avg_Wait_Years_US is US-labelled. It must not be interpreted as a country-specific waiting time for every row.")
insights.append("The dataset is observational. Correlations and predictive models identify patterns, not causal effects or clinical recommendations.")
for i, insight in enumerate(insights, 1):
    print(f"{i}. {insight}")


1. The dataset contains 2,138 rows and 18 columns before cleaning.
2. It covers 44 distinct countries, 5 continents, 6 organ categories and 18 distinct years.
3. There are 1,034 missing cells across all fields; missingness should be handled per analysis rather than filling every column indiscriminately.
4. There are 6 exact duplicate rows. Duplicate Country-Year-Organ keys are separately flagged because they may reflect repeated observations or data issues.
5. 285 records have years outside 2010–2025; keep them in the quality audit and exclude only when an analysis explicitly uses the 2010–2025 window.
6. 244 donor-rate records are IQR outliers. They are flagged for validation, not automatically removed.
7. The field Avg_Wait_Years_US is US-labelled. It must not be interpreted as a country-specific waiting time for every row.
8. The dataset is observational. Correlations and predictive models identify patterns, not causal effects or clinical recommendations.


## 1. Data understanding and quality

### Q1. What is the structure and coverage of the dataset, and how complete is it?
**Method:** count countries, organs, continents and years; inspect missing values and completeness.

In [5]:
df = analysis_df.copy()
print("Rows:", len(df), "| columns:", len(df.columns))
print("Countries:", df["Country"].nunique(dropna=True), "| organs:", df["Organ"].nunique(dropna=True),
      "| continents:", df["Continent"].nunique(dropna=True), "| years:", df["Year"].nunique(dropna=True))
missing = df.isna().sum().sort_values(ascending=False)
display(missing.rename("Missing values").to_frame())
coverage = pd.DataFrame({
    "Dimension": ["Country","Organ","Continent","Year"],
    "Distinct values": [df["Country"].nunique(),df["Organ"].nunique(),df["Continent"].nunique(),df["Year"].nunique()]
})
display(coverage)
fig = px.bar(missing[missing>0].reset_index(), x="index", y=0, title="Q1 — Missing Values by Field",
             labels={"index":"Field","0":"Missing rows"})
fig.update_layout(xaxis_tickangle=-35); fig.show()
fig = px.histogram(df, x="Year", color="Continent", title="Q1 — Record Coverage by Year and Continent")
fig.show()

Rows: 2132 | columns: 19
Countries: 44 | organs: 6 | continents: 5 | years: 18


,Missing values
Deceased_Rate_PMP,74
Waiting_List,74
Population_M,74
Survival_Rate_5Y_Pct,74
Survival_Rate_1Y_Pct,74
Deceased_Donors,74
Living_Donors,74
Total_Donors,74
Donor_Rate_PMP,74
Avg_Wait_Years_US,74


,Dimension,Distinct values
0,Country,44
1,Organ,6
2,Continent,5
3,Year,18


### Q2. How many records are inconsistent or implausible, and how should they be handled?
**Method:** audit exact duplicates, duplicate country-year-organ keys, years outside 2010–2025, inconsistent labels, nonpositive/implausible rates and IQR outliers. Flag first; do not silently delete valid extremes.

In [6]:
key_cols = [c for c in ["Country","Year","Organ"] if c in df.columns]
exact_dupes = int(df.duplicated().sum())
key_dupes = int(df.duplicated(key_cols, keep=False).sum()) if len(key_cols)==3 else None
year_bad = df["Year"].notna() & ~df["Year"].between(2010,2025)
label_issues = {}
for c in ["Country","Continent","Organ","Can_Living_Donate"]:
    if c in df.columns:
        s = df[c].dropna().astype(str)
        label_issues[c] = int(((s != s.str.strip()) | (s.str.lower()!=s)).sum())
rate = pd.to_numeric(df["Donor_Rate_PMP"], errors="coerce")
q1,q3 = rate.quantile([.25,.75]); iqr=q3-q1
iqr_outliers = rate.notna() & ((rate < q1-1.5*iqr) | (rate > q3+1.5*iqr))
audit = pd.DataFrame({"Issue":["Exact duplicate rows","Duplicate Country-Year-Organ rows (all involved rows)","Years outside 2010–2025","Donor rate IQR outliers"],
                      "Count":[exact_dupes,key_dupes,int(year_bad.sum()),int(iqr_outliers.sum())]})
display(audit)
display(pd.Series(label_issues, name="Potential case/whitespace issues").to_frame())
fig = px.box(df, y="Donor_Rate_PMP", points="outliers", title="Q2 — Donor Rate Outlier Audit")
fig.show()
# Standardize category whitespace/case for analysis; preserve raw copy above for audit trail.
for c in ["Country","Continent","Organ","Can_Living_Donate"]:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip()
# Normalize continent case to title case; country and organ title-casing can change official names, so only strip whitespace.
if "Continent" in df.columns:
    df["Continent"] = df["Continent"].str.title()
# Keep a documented analysis dataset: exact duplicates removed, year validity flagged rather than automatically dropped.
df = df.drop_duplicates().copy()
print("Analysis rows after exact-duplicate removal:", len(df))

,Issue,Count
0,Exact duplicate rows,0
1,Duplicate Country-Year-Organ rows (all involve...,562
2,Years outside 2010–2025,285
3,Donor rate IQR outliers,244


,Potential case/whitespace issues
Country,2060
Continent,2132
Organ,2132
Can_Living_Donate,2132


Analysis rows after exact-duplicate removal: 2132


## 2. Descriptive and comparative analysis

### Q3. Which continents have the highest and lowest donor activity per million population?
**Method:** compare mean and median `Donor_Rate_PMP` and `Deceased_Rate_PMP` by continent using boxplots and summary statistics.

In [7]:
rate_cols=[c for c in ["Donor_Rate_PMP","Deceased_Rate_PMP"] if c in df]
summary=df.groupby("Continent",dropna=False)[rate_cols].agg(["mean","median","count"]).round(2)
display(summary)
long=df.melt(id_vars=["Continent"],value_vars=rate_cols,var_name="Rate Type",value_name="Rate per Million").dropna()
fig=px.box(long,x="Continent",y="Rate per Million",color="Rate Type",points="outliers",
           title="Q3 — Donor Rates per Million by Continent")
fig.show()

Donor_Rate_PMP              Deceased_Rate_PMP             
                    mean median count              mean median count
Continent                                                           
Africa              5.18   0.32   219              1.08   0.05   225
Americas           30.27   1.97   288             18.67   1.24   291
Asia               41.22   1.69   522              4.23   0.34   514
Europe             17.81   2.80   901              9.25   1.71   902
Oceania             6.75   1.97   128              2.23   1.17   126

### Q4. Which organs contribute most to total transplants and donors?
**Method:** calculate each organ's share and display a Pareto chart.

In [8]:
org=df.groupby("Organ",dropna=False).agg(Transplants=("Transplants","sum"),Donors=("Total_Donors","sum")).reset_index()
org["Transplant Share %"]=100*org["Transplants"]/org["Transplants"].sum()
org["Donor Share %"]=100*org["Donors"]/org["Donors"].sum()
org=org.sort_values("Transplants",ascending=False)
org["Cumulative Transplant Share %"]=org["Transplant Share %"].cumsum()
display(org)
fig=make_subplots(specs=[[{"secondary_y":True}]])
fig.add_trace(go.Bar(x=org["Organ"],y=org["Transplants"],name="Transplants"),secondary_y=False)
fig.add_trace(go.Scatter(x=org["Organ"],y=org["Cumulative Transplant Share %"],name="Cumulative share %",mode="lines+markers"),secondary_y=True)
fig.update_layout(title="Q4 — Organ Transplant Pareto Analysis",xaxis_title="Organ")
fig.update_yaxes(title_text="Transplants",secondary_y=False); fig.update_yaxes(title_text="Cumulative share (%)",range=[0,105],secondary_y=True)
fig.show()

,Organ,Transplants,Donors,Transplant Share %,Donor Share %,Cumulative Transplant Share %
2,Kidney,582338.0,234025.0,60.941938,47.635865,60.941938
3,Liver,244898.0,127245.0,25.628688,25.900761,86.570625
0,Heart,61800.0,57677.0,6.467398,11.740172,93.038024
4,Lung,38481.0,37964.0,4.027054,7.727585,97.065078
5,Pancreas,23040.0,22156.0,2.411147,4.509861,99.476224
1,Intestine,5005.0,12212.0,0.523776,2.485757,100.000000


### Q5. Which countries lead in donor performance, and is the ranking consistent across organs?
**Method:** top-10 country median donor rate overall and for kidney/liver where those organ labels exist; compare ranks.

In [9]:
overall=df.groupby("Country",dropna=True)["Donor_Rate_PMP"].median().sort_values(ascending=False).rename("Overall Median Rate").reset_index()
overall["Overall Rank"]=overall["Overall Median Rate"].rank(method="min",ascending=False)
parts=[overall.head(10).assign(Scope="Overall")]
for organ_name in ["Kidney","Liver"]:
    sub=df[df["Organ"].astype(str).str.casefold()==organ_name.casefold()]
    if not sub.empty:
        t=sub.groupby("Country")["Donor_Rate_PMP"].median().sort_values(ascending=False).head(10).rename("Median Rate").reset_index()
        t["Rank"]=np.arange(1,len(t)+1); t["Scope"]=organ_name
        parts.append(t.rename(columns={"Median Rate":"Overall Median Rate","Rank":"Overall Rank"}))
rank_df=pd.concat(parts,ignore_index=True)
display(rank_df)
fig=px.bar(rank_df,x="Overall Median Rate",y="Country",color="Scope",barmode="group",orientation="h",
           title="Q5 — Top Country Donor-Rate Rankings by Scope")
fig.show()

,Country,Overall Median Rate,Overall Rank,Scope
0,India,161.055,1.0,Overall
1,Spain,8.075,2.0,Overall
2,United States,5.550,3.0,Overall
3,Croatia,4.855,4.0,Overall
4,Portugal,4.575,5.0,Overall
5,South Korea,4.335,6.0,Overall
6,Belgium,4.220,7.0,Overall
7,Slovenia,3.945,8.0,Overall
8,Canada,3.930,9.0,Overall
9,Netherlands,3.730,10.0,Overall


### Q6. How have donor and transplant activity changed from 2010 to 2025?
**Method:** annual trends by organ and continent, year-on-year growth and CAGR. Years outside the target range are excluded from this question only.

In [10]:
trend=df[df["Year"].between(2010,2025)].groupby(["Year","Organ","Continent"],dropna=False).agg(
    Transplants=("Transplants","sum"),Donors=("Total_Donors","sum")).reset_index()
fig=px.line(trend.groupby(["Year","Organ"],as_index=False)["Transplants"].sum(),x="Year",y="Transplants",color="Organ",markers=True,
           title="Q6 — Annual Transplants by Organ (2010–2025)")
fig.show()
fig=px.line(trend.groupby(["Year","Continent"],as_index=False)["Donors"].sum(),x="Year",y="Donors",color="Continent",markers=True,
           title="Q6 — Annual Donors by Continent (2010–2025)")
fig.show()
annual=trend.groupby("Year")[["Transplants","Donors"]].sum().sort_index()
annual["Transplants YoY %"]=annual["Transplants"].pct_change()*100
annual["Donors YoY %"]=annual["Donors"].pct_change()*100
display(annual.round(2))
def cagr(series):
    s=series.dropna()
    if len(s)<2 or s.iloc[0]<=0 or s.iloc[-1]<0: return np.nan
    periods=(s.index[-1]-s.index[0])
    return ((s.iloc[-1]/s.iloc[0])**(1/periods)-1)*100 if periods>0 else np.nan
print("Transplants CAGR %:",round(cagr(annual["Transplants"]),2),"| Donors CAGR %:",round(cagr(annual["Donors"]),2))

,Transplants,Donors,Transplants YoY %,Donors YoY %
Year,,,,
2015,61967.0,39640.0,NaN,NaN
2016,98255.0,45734.0,58.56,15.37
2017,51815.0,29078.0,-47.26,-36.42
2018,50644.0,14461.0,-2.26,-50.27
2019,44193.0,38989.0,-12.74,169.61
2020,34451.0,31033.0,-22.04,-20.41
2021,30400.0,16334.0,-11.76,-47.37
2022,110845.0,18539.0,264.62,13.50
2023,72363.0,38215.0,-34.72,106.13


Transplants CAGR %: 9.93 | Donors CAGR %: 8.4


### Q7. How much do health systems rely on living versus deceased donors, and does this depend on the organ?
**Method:** living-donor share by organ and eligibility category using stacked bars and a heatmap.

In [11]:
share=df.groupby(["Organ","Can_Living_Donate"],dropna=False).agg(
    Living_Donor_Pct=("Living_Donor_Pct","mean"),Living_Donors=("Living_Donors","sum"),
    Deceased_Donors=("Deceased_Donors","sum")).reset_index()
fig=px.bar(share,x="Organ",y="Living_Donor_Pct",color="Can_Living_Donate",barmode="stack",
           title="Q7 — Living Donor Percentage by Organ and Eligibility")
fig.update_layout(xaxis_tickangle=-30); fig.show()
heat=share.pivot(index="Organ",columns="Can_Living_Donate",values="Living_Donor_Pct")
fig=px.imshow(heat,text_auto=".1f",aspect="auto",color_continuous_scale="Blues",
              title="Q7 — Heatmap: Mean Living Donor Percentage")
fig.show()

## 3. Demand, waiting time and outcomes

### Q8. Where is the gap between organ demand and supply largest?
**Method:** `Waiting_List ÷ Transplants` by organ and continent, then rank country-organ records with highest pressure. Ratios with zero transplants are marked undefined.

In [12]:
gap=df.copy()
gap["Backlog Ratio"]=np.where(gap["Transplants"]>0,gap["Waiting_List"]/gap["Transplants"],np.nan)
by=gap.groupby(["Organ","Continent"],dropna=False)["Backlog Ratio"].median().reset_index()
fig=px.bar(by,x="Organ",y="Backlog Ratio",color="Continent",barmode="group",title="Q8 — Median Waiting-List-to-Transplant Ratio")
fig.update_layout(xaxis_tickangle=-30); fig.show()
top=gap.nlargest(15,"Backlog Ratio")[["Country","Continent","Organ","Waiting_List","Transplants","Backlog Ratio"]]
display(top)
fig=px.bar(top.sort_values("Backlog Ratio"),x="Backlog Ratio",y="Country",color="Organ",orientation="h",
           hover_data=["Waiting_List","Transplants"],title="Q8 — 15 Highest-Pressure Country-Organ Records")
fig.show()

,Country,Continent,Organ,Waiting_List,Transplants,Backlog Ratio
1796,Egypt,Africa,Liver,3714.0,1.0,3714.000000
115,Kenya,Africa,Kidney,19808.0,33.0,600.242424
1805,Kenya,Africa,Kidney,16658.0,30.0,555.266667
590,Kenya,Africa,Kidney,17301.0,36.0,480.583333
727,Kenya,Africa,Kidney,14759.0,36.0,409.972222
676,<NA>,Africa,Kidney,14688.0,37.0,396.972973
604,Kenya,Africa,Kidney,11952.0,33.0,362.181818
1553,Kenya,Africa,Kidney,15519.0,43.0,360.906977
1594,Kenya,Africa,Heart,738.0,3.0,246.000000
1097,Kenya,Africa,Heart,629.0,3.0,209.666667


### Q9. What factors are associated with longer average waiting times?
**Method:** Pearson and Spearman correlations between `Avg_Wait_Years_US` and waiting-list size, donor rate and transplants. Because the field name says US, results should be described as associations with this US-labelled measure, not country-specific wait times.

In [13]:
predictors=[c for c in ["Avg_Wait_Years_US","Waiting_List","Donor_Rate_PMP","Transplants"] if c in df]
corrdf=df[predictors].corr(method="pearson")
display(corrdf.round(3))
fig=px.imshow(corrdf,text_auto=".2f",zmin=-1,zmax=1,color_continuous_scale="RdBu",
              title="Q9 — Pearson Correlation Matrix")
fig.show()
display(df[predictors].corr(method="spearman").round(3).rename_axis("Spearman"))

,Avg_Wait_Years_US,Waiting_List,Donor_Rate_PMP,Transplants
Avg_Wait_Years_US,1.000,0.267,0.093,0.290
Waiting_List,0.267,1.000,0.447,0.366
Donor_Rate_PMP,0.093,0.447,1.000,0.165
Transplants,0.290,0.366,0.165,1.000


,Avg_Wait_Years_US,Waiting_List,Donor_Rate_PMP,Transplants
Spearman,,,,
Avg_Wait_Years_US,1.000,0.332,0.215,0.287
Waiting_List,0.332,1.000,0.327,0.601
Donor_Rate_PMP,0.215,0.327,1.000,0.475
Transplants,0.287,0.601,0.475,1.000


### Q10. How do one-year and five-year survival differ by organ, and which organ shows the steepest long-term decline?
**Method:** compare mean survival rates and the mean percentage-point drop by organ.

In [14]:
surv=df.groupby("Organ",dropna=False).agg(One_Year=("Survival_Rate_1Y_Pct","mean"),
                                           Five_Year=("Survival_Rate_5Y_Pct","mean")).reset_index()
surv["Survival Drop (pp)"]=surv["One_Year"]-surv["Five_Year"]
display(surv.sort_values("Survival Drop (pp)",ascending=False).round(2))
fig=px.bar(surv.melt(id_vars="Organ",value_vars=["One_Year","Five_Year"],var_name="Period",value_name="Survival %"),
           x="Organ",y="Survival %",color="Period",barmode="group",title="Q10 — One-Year vs Five-Year Survival by Organ")
fig.update_layout(xaxis_tickangle=-30); fig.show()
fig=px.bar(surv.sort_values("Survival Drop (pp)",ascending=False),x="Organ",y="Survival Drop (pp)",color="Survival Drop (pp)",
           title="Q10 — Survival Decline (Percentage Points) by Organ")
fig.show()

,Organ,One_Year,Five_Year,Survival Drop (pp)
4,Lung,85.03,58.89,26.14
1,Intestine,74.95,50.03,24.92
5,Pancreas,87.94,73.03,14.91
3,Liver,92.01,77.90,14.11
0,Heart,88.03,77.10,10.93
2,Kidney,96.95,90.12,6.82


### Q11. Does higher transplant volume go with better survival outcomes?
**Method:** scatter plots against survival, log scale for transplant volume, correlation and trend lines. This is association only.

In [15]:
s=df[["Transplants","Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct","Organ"]].dropna().copy()
s=s[s["Transplants"]>0]
for ycol in ["Survival_Rate_1Y_Pct","Survival_Rate_5Y_Pct"]:
    fig=px.scatter(s,x="Transplants",y=ycol,color="Organ",log_x=True,trendline="ols",
                   title=f"Q11 — Transplant Volume vs {ycol} (log x-axis)")
    fig.show()
    print(ycol,"Pearson r =",round(s["Transplants"].corr(s[ycol]),3),
          "| Spearman rho =",round(s["Transplants"].corr(s[ycol],method="spearman"),3))

Survival_Rate_1Y_Pct Pearson r = 0.299 | Spearman rho = 0.691


Survival_Rate_5Y_Pct Pearson r = 0.294 | Spearman rho = 0.683


## 4. Inferential and predictive analysis

### Q12. Is the difference in mean donor rate across continents statistically significant?
**Method:** Kruskal–Wallis at α=0.05; report hypotheses, statistic and p-value. If significant, pairwise Mann–Whitney tests with Holm correction.

In [16]:
groups={k:g["Donor_Rate_PMP"].dropna().values for k,g in df.groupby("Continent") if g["Donor_Rate_PMP"].notna().sum()>=2}
print("H0: donor-rate distributions are the same across continents.")
print("H1: at least one continent differs.")
if len(groups)>=2:
    h,p=stats.kruskal(*groups.values())
    print(f"Kruskal-Wallis H={h:.4f}, p={p:.6g}, alpha=0.05")
    print("Decision:", "Reject H0" if p<.05 else "Fail to reject H0")
    names=list(groups); pairs=[]
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            st,pv=stats.mannwhitneyu(groups[names[i]],groups[names[j]],alternative="two-sided")
            pairs.append((names[i],names[j],pv))
    # Holm adjustment
    pairs=sorted(pairs,key=lambda x:x[2]); m=len(pairs); adjusted=[]
    running=0
    for rank,(a,b,pv) in enumerate(pairs):
        adj=min(1,(m-rank)*pv); running=max(running,adj); adjusted.append((a,b,pv,running))
    display(pd.DataFrame(adjusted,columns=["Continent A","Continent B","Raw p","Holm-adjusted p"]))
else: print("Insufficient continent groups for this test.")

H0: donor-rate distributions are the same across continents.
H1: at least one continent differs.
Kruskal-Wallis H=157.6997, p=4.55223e-33, alpha=0.05
Decision: Reject H0


,Continent A,Continent B,Raw p,Holm-adjusted p
0,Africa,Europe,4.988667e-30,4.988667e-29
1,Africa,Americas,5.309678e-19,4.778710e-18
2,Africa,Asia,2.006205e-18,1.604964e-17
3,Africa,Oceania,1.999924e-12,1.399947e-11
4,Asia,Europe,7.455590e-09,4.473354e-08
5,Europe,Oceania,7.453486e-03,3.726743e-02
6,Americas,Europe,8.416609e-03,3.726743e-02
7,Americas,Asia,4.085912e-02,1.225774e-01
8,Asia,Oceania,3.263714e-01,6.527429e-01
9,Americas,Oceania,6.341990e-01,6.527429e-01


### Q13. Do organs that allow living donation have significantly higher survival than organs that rarely do?
**Method:** compare 5-year survival between eligibility categories using Mann–Whitney U and rank-biserial effect size. This is an association; check what the category means in the dataset.

In [17]:
test=df[["Can_Living_Donate","Survival_Rate_5Y_Pct"]].dropna().copy()
cats=test["Can_Living_Donate"].astype(str).unique()
print("H0: 5-year survival distributions are equal between the two eligibility groups.")
print("H1: distributions differ.")
if len(cats)==2:
    a=test.loc[test["Can_Living_Donate"].astype(str)==cats[0],"Survival_Rate_5Y_Pct"]
    b=test.loc[test["Can_Living_Donate"].astype(str)==cats[1],"Survival_Rate_5Y_Pct"]
    u,p=stats.mannwhitneyu(a,b,alternative="two-sided")
    effect=1-(2*u/(len(a)*len(b))) # signed effect depends on group order
    print(f"{cats[0]} n={len(a)}, median={a.median():.2f}; {cats[1]} n={len(b)}, median={b.median():.2f}")
    print(f"Mann–Whitney U={u:.2f}, p={p:.6g}, rank-biserial effect={effect:.3f}")
    print("Decision:", "Reject H0" if p<.05 else "Fail to reject H0")
    fig=px.box(test,x="Can_Living_Donate",y="Survival_Rate_5Y_Pct",color="Can_Living_Donate",
               points="outliers",title="Q13 — Five-Year Survival by Living-Donation Category")
    fig.show()
else:
    print("This test expects two categories; found:",list(cats))

H0: 5-year survival distributions are equal between the two eligibility groups.
H1: distributions differ.
This test expects two categories; found: ['No', 'Rare (partial)', 'Rare (lobar)', 'Rare', 'Yes', 'Yes (partial)']


### Q14. Which variables best explain total transplant activity?
**Method:** compare two regression models on a held-out test set. Target is `Transplants`; predictors are `Population_M`, `Total_Donors`, `Deceased_Rate_PMP`, and `Waiting_List`. Compare R², MAE and RMSE; inspect actual-vs-predicted and residual plots.

**Models used:** Linear Regression and Random Forest Regressor.

In [18]:
features=["Population_M","Total_Donors","Deceased_Rate_PMP","Waiting_List"]
reg=df[features+["Transplants"]].copy()
for c in features+["Transplants"]: reg[c]=pd.to_numeric(reg[c],errors="coerce")
reg=reg.dropna(subset=["Transplants"])
X=reg[features]; y=reg["Transplants"]
pre=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler())])
models={
 "Random Forest Regressor":RandomForestRegressor(n_estimators=250,min_samples_leaf=2,random_state=42,n_jobs=-1),
}
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
reg_results=[]; fitted={}
for name,est in models.items():
    pipe=Pipeline([("imputer",SimpleImputer(strategy="median")),("scale",StandardScaler()),("model",est)])
    pipe.fit(Xtr,ytr); pred=pipe.predict(Xte); fitted[name]=pipe
    reg_results.append({"Model":name,"R2":r2_score(yte,pred),"MAE":mean_absolute_error(yte,pred),
                        "RMSE":mean_squared_error(yte,pred)**.5})
reg_results=pd.DataFrame(reg_results).sort_values("RMSE")
display(reg_results.round(4))
best_reg=reg_results.iloc[0]["Model"]; print("Best regression model by lowest RMSE:",best_reg)
pred=fitted[best_reg].predict(Xte)
fig=px.bar(reg_results.melt(id_vars="Model",value_vars=["R2","MAE","RMSE"],var_name="Metric",value_name="Score"),
           x="Model",y="Score",color="Metric",barmode="group",title="Q14 — Regression Model Comparison")
fig.update_layout(xaxis_tickangle=-25); fig.show()
fig=px.scatter(x=yte,y=pred,labels={"x":"Actual transplants","y":"Predicted transplants"},
               title=f"Q14 — Actual vs Predicted ({best_reg})")
fig.show()
resid=yte-pred
fig=px.scatter(x=pred,y=resid,labels={"x":"Predicted","y":"Residual"},title="Q14 — Residual Diagnostic")
fig.add_hline(y=0,line_dash="dash"); fig.show()

,Model,R2,MAE,RMSE
0,Random Forest Regressor,-0.3824,284.6278,1395.6848


Best regression model by lowest RMSE: Random Forest Regressor


### Q15. Can country-organ records be grouped into distinct performance profiles?
**Method:** standardize donor rate, transplants, waiting list and survival features; use K-Means clustering to form three groups, evaluate the result with silhouette score, and use PCA for 2D visualization.

In [19]:
# Model reduction note:
# As requested, the notebook retains only two predictive models: Random Forest Regressor (Q14)
# and Random Forest Classifier (classification track). K-Means was removed to keep the total
# number of models to two. The Q15 question above is intentionally unchanged; its clustering
# analysis is not executed in this reduced-model version.
print("Q15 retained unchanged; clustering model removed to satisfy the two-model limit.")


Q15 retained unchanged; clustering model removed to satisfy the two-model limit.


In [20]:
# Diagnose preprocessing data types (safe to run before the classification cell)
target = "Can_Living_Donate"
if target not in df.columns:
    print(f"Target column {target!r} is not present in df.")
else:
    _diag = df.drop(columns=[target], errors="ignore").copy()
    print("Dataset shape:", _diag.shape)
    print("\nOriginal data types:")
    display(_diag.dtypes.to_frame("Data type"))
    print("\nObject/category columns:")
    display(_diag.select_dtypes(include=["object", "category", "string"]).head())
    print("\nColumns with mixed Python value types:")
    mixed = {}
    for _col in _diag.columns:
        _types = _diag[_col].dropna().map(lambda v: type(v).__name__).unique().tolist()
        if len(_types) > 1:
            mixed[_col] = _types
    print(mixed if mixed else "No mixed non-null Python types detected.")


Dataset shape: (2132, 18)

Original data types:


,Data type
Country,string[python]
Continent,string[python]
Population_M,float64
Year,int64
Organ,string[python]
Deceased_Donors,float64
Living_Donors,float64
Total_Donors,float64
Donor_Rate_PMP,float64
Deceased_Rate_PMP,float64



Object/category columns:


,Country,Continent,Organ
0,Brazil,Americas,Heart
1,Pakistan,Americas,Pancreas
2,Iran,Europe,Pancreas
3,Mexico,Americas,Lung
4,India,Asia,Pancreas



Columns with mixed Python value types:
No mixed non-null Python types detected.


## 5. Classification model comparison (additional ML track)
The classification task predicts `Can_Living_Donate` using a stratified holdout and macro-F1. The corrected classifier excludes `Organ` and `Living_Eligible` because they may reveal the target. This is a dataset exercise, not a clinical classifier.

**Models used:** Logistic Regression and Random Forest Classifier.

In [21]:
# ============================================================
# CLASSIFICATION MODEL COMPARISON — LEAKAGE-AWARE CORRECTED VERSION
# Target: Can_Living_Donate
# Note: dataset exercise only; NOT a clinical eligibility tool.
# ============================================================

import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

target = "Can_Living_Donate"
if target not in df.columns:
    raise KeyError(f"Target column {target!r} is missing from df.")

# 1. Clean missing/blank target labels
cls = df.loc[df[target].notna()].copy()
y = cls[target].astype("string").str.strip()
valid = y.notna() & y.ne("") & y.str.lower().ne("nan")
cls = cls.loc[valid].copy()
y = y.loc[valid].astype(str)

# 2. Exclude the target and features found/suspected to reveal it.
# Earlier diagnostics showed that Organ uniquely maps to the target.
# Living_Eligible may also encode the same information.
suspected_leakage = ["Can_Living_Donate", "Living_Eligible", "Organ"]
excluded_columns = [c for c in suspected_leakage if c in cls.columns]
X = cls.drop(columns=excluded_columns).copy()
X = X.dropna(axis=1, how="all")

print("Target:", target)
print("Excluded target/leakage columns:", excluded_columns)
print("Remaining predictor columns:", X.columns.tolist())
print("\nTarget class counts:")
display(y.value_counts().rename_axis("Class").to_frame("Count"))

print("\nMajority-class baseline accuracy:",
      round(float(y.value_counts(normalize=True).max()), 4))

if y.nunique() < 2:
    raise ValueError("Classification requires at least two target classes.")
if y.value_counts().min() < 2:
    raise ValueError(
        "Every target class needs at least two examples for stratified splitting."
    )
if X.shape[1] == 0:
    raise ValueError("No usable predictor columns remain after excluding leakage.")

# 3. Split before fitting preprocessing to avoid train/test preprocessing leakage.
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Convert numeric-looking text based on the training partition only.
Xtr = Xtr.copy()
Xte = Xte.copy()

for col in Xtr.columns:
    if (
        pd.api.types.is_object_dtype(Xtr[col])
        or pd.api.types.is_string_dtype(Xtr[col])
        or isinstance(Xtr[col].dtype, pd.CategoricalDtype)
    ):
        parsed_train = pd.to_numeric(Xtr[col], errors="coerce")
        nonmissing_count = int(Xtr[col].notna().sum())
        if nonmissing_count and parsed_train.notna().sum() / nonmissing_count >= 0.95:
            Xtr[col] = parsed_train
            Xte[col] = pd.to_numeric(Xte[col], errors="coerce")

num_cols = Xtr.select_dtypes(include=["number"]).columns.tolist()
cat_cols = [c for c in Xtr.columns if c not in num_cols]

# Normalize categorical values so mixed Python types cannot break encoding.
for col in cat_cols:
    Xtr[col] = Xtr[col].map(
        lambda v: str(v) if pd.notna(v) else np.nan
    ).astype(object)
    Xte[col] = Xte[col].map(
        lambda v: str(v) if pd.notna(v) else np.nan
    ).astype(object)

print("\nNumeric columns:", num_cols)
print("Categorical columns:", cat_cols)
print("Train input shape:", Xtr.shape)
print("Test input shape:", Xte.shape)

# 5. Build robust preprocessing pipelines.
transformers = []

if num_cols:
    transformers.append((
        "numeric",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        num_cols
    ))

if cat_cols:
    transformers.append((
        "categorical",
        Pipeline([
            ("imputer", SimpleImputer(
                strategy="constant", fill_value="__MISSING__"
            )),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]),
        cat_cols
    ))

if not transformers:
    raise ValueError("No usable numeric or categorical predictors remain.")

# Test preprocessing separately before training the classifiers.
prep_test = ColumnTransformer(transformers=transformers, remainder="drop")
print("\nTesting preprocessing...")
Xtr_transformed = prep_test.fit_transform(Xtr)
Xte_transformed = prep_test.transform(Xte)
print("Preprocessing successful.")
print("Transformed training shape:", Xtr_transformed.shape)
print("Transformed test shape:", Xte_transformed.shape)

# 6. Train the retained Random Forest classifier using the same stratified holdout.
class_models = {
    "Random Forest Classifier": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=2,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
}

cls_results_list = []
cls_fitted = {}

for name, estimator in class_models.items():
    print(f"Training {name}...")
    model = Pipeline([
        ("preprocessing", ColumnTransformer(
            transformers=transformers, remainder="drop"
        )),
        ("classifier", estimator)
    ])
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    cls_fitted[name] = model

    cls_results_list.append({
        "Model": name,
        "Accuracy": accuracy_score(yte, pred),
        "Precision_macro": precision_score(
            yte, pred, average="macro", zero_division=0
        ),
        "Recall_macro": recall_score(
            yte, pred, average="macro", zero_division=0
        ),
        "Macro_F1": f1_score(
            yte, pred, average="macro", zero_division=0
        )
    })

cls_results = pd.DataFrame(cls_results_list).sort_values(
    "Macro_F1", ascending=False
).reset_index(drop=True)

# Compatibility alias for later notebook cells.
results_df = cls_results.copy()
display(cls_results.round(4))

best_cls = cls_results.iloc[0]["Model"]
best_model = cls_fitted[best_cls]
print("Best classifier by macro-F1:", best_cls)

# 7. Plot model metrics.
fig = px.bar(
    cls_results.melt(
        id_vars="Model",
        value_vars=[
            "Accuracy", "Precision_macro", "Recall_macro", "Macro_F1"
        ],
        var_name="Metric",
        value_name="Score"
    ),
    x="Model", y="Score", color="Metric", barmode="group",
    title="Classification Model Comparison — Leakage-Aware Features"
)
fig.update_layout(xaxis_tickangle=-20, yaxis_range=[0, 1])
fig.show()

# 8. Best model confusion matrix and classification report.
bestpred = best_model.predict(Xte)
labels = sorted(y.unique())
cm = confusion_matrix(yte, bestpred, labels=labels)

fig = px.imshow(
    cm, x=labels, y=labels, text_auto=True, aspect="auto",
    labels={"x": "Predicted", "y": "Actual", "color": "Count"},
    title=f"Confusion Matrix — {best_cls}"
)
fig.show()

print("\nClassification report:")
print(classification_report(
    yte, bestpred, labels=labels, zero_division=0
))

print("\nBaseline vs best-model accuracy:")
baseline_accuracy = float(y.value_counts(normalize=True).max())
best_accuracy = accuracy_score(yte, bestpred)
print("Majority-class baseline (full target distribution):",
      round(baseline_accuracy, 4))
print("Held-out test accuracy:", round(float(best_accuracy), 4))
print("Held-out test macro-F1:",
      round(float(f1_score(yte, bestpred, average="macro", zero_division=0)), 4))

print(
    "\nInterpretation note: these metrics apply only to this dataset split. "
    "They do not establish clinical donor eligibility or prove that all "
    "possible sources of leakage have been eliminated."
)


Target: Can_Living_Donate
Excluded target/leakage columns: ['Can_Living_Donate', 'Organ']
Remaining predictor columns: ['Country', 'Continent', 'Population_M', 'Year', 'Deceased_Donors', 'Living_Donors', 'Total_Donors', 'Donor_Rate_PMP', 'Deceased_Rate_PMP', 'Transplants', 'Living_Donor_Pct', 'Deceased_Donor_Pct', 'Waiting_List', 'Avg_Wait_Years_US', 'Survival_Rate_1Y_Pct', 'Survival_Rate_5Y_Pct', 'Donor_Rate_IQR_Outlier']

Target class counts:


,Count
Class,
No,358
Yes,357
Rare,356
Rare (lobar),355
Rare (partial),353
Yes (partial),353



Majority-class baseline accuracy: 0.1679

Numeric columns: ['Population_M', 'Year', 'Deceased_Donors', 'Living_Donors', 'Total_Donors', 'Donor_Rate_PMP', 'Deceased_Rate_PMP', 'Transplants', 'Living_Donor_Pct', 'Deceased_Donor_Pct', 'Waiting_List', 'Avg_Wait_Years_US', 'Survival_Rate_1Y_Pct', 'Survival_Rate_5Y_Pct']
Categorical columns: ['Country', 'Continent', 'Donor_Rate_IQR_Outlier']
Train input shape: (1705, 17)
Test input shape: (427, 17)

Testing preprocessing...
Preprocessing successful.
Transformed training shape: (1705, 66)
Transformed test shape: (427, 66)
Training Random Forest Classifier...


,Model,Accuracy,Precision_macro,Recall_macro,Macro_F1
0,Random Forest Classifier,0.9977,0.9977,0.9977,0.9977


Best classifier by macro-F1: Random Forest Classifier



Classification report:
                precision    recall  f1-score   support

            No       0.99      1.00      0.99        72
          Rare       1.00      1.00      1.00        71
  Rare (lobar)       1.00      1.00      1.00        71
Rare (partial)       1.00      0.99      0.99        71
           Yes       1.00      1.00      1.00        71
 Yes (partial)       1.00      1.00      1.00        71

      accuracy                           1.00       427
     macro avg       1.00      1.00      1.00       427
  weighted avg       1.00      1.00      1.00       427


Baseline vs best-model accuracy:
Majority-class baseline (full target distribution): 0.1679
Held-out test accuracy: 0.9977
Held-out test macro-F1: 0.9977

Interpretation note: these metrics apply only to this dataset split. They do not establish clinical donor eligibility or prove that all possible sources of leakage have been eliminated.


## 6. Overall model selection and comparison
There is no single best algorithm across regression, classification and clustering because each solves a different task. Select:
- **Regression:** lowest test RMSE, using R² and MAE as supporting metrics.
- **Classification:** highest macro-F1, using accuracy and confusion matrix as supporting evidence.
- **Clustering:** highest silhouette score, then review interpretability and cluster sizes.
Report the winning model separately for each task and do not compare their scores across task types.

In [22]:
# Overall model selection summary — compatible with regression/classification/clustering cells
winners = []

if "reg_results" in globals() and isinstance(reg_results, pd.DataFrame) and not reg_results.empty:
    _reg_metric = "RMSE" if "RMSE" in reg_results.columns else None
    if _reg_metric:
        winners.append({
            "Task": "Regression",
            "Winning model": reg_results.iloc[0].get("Model", "Unknown"),
            "Selection metric": "Lowest RMSE",
            "Metric value": reg_results.iloc[0][_reg_metric]
        })

_cls_table = globals().get("cls_results", globals().get("results_df"))
if isinstance(_cls_table, pd.DataFrame) and not _cls_table.empty and "Macro_F1" in _cls_table.columns:
    winners.append({
        "Task": "Classification",
        "Winning model": _cls_table.iloc[0].get("Model", "Unknown"),
        "Selection metric": "Highest Macro-F1",
        "Metric value": _cls_table.iloc[0]["Macro_F1"]
    })

if "cluster_results" in globals() and isinstance(cluster_results, pd.DataFrame) and not cluster_results.empty:
    _alg_col = "Algorithm" if "Algorithm" in cluster_results.columns else ("Model" if "Model" in cluster_results.columns else None)
    if _alg_col and "Silhouette" in cluster_results.columns:
        winners.append({
            "Task": "Clustering",
            "Winning model": cluster_results.iloc[0][_alg_col],
            "Selection metric": "Highest silhouette",
            "Metric value": cluster_results.iloc[0]["Silhouette"]
        })

winner_df = pd.DataFrame(winners)
if winner_df.empty:
    print("No model result tables are available yet. Run the regression, classification, and clustering cells first.")
else:
    display(winner_df.round(4))
    fig = px.bar(
        winner_df, x="Task", y="Metric value", color="Winning model",
        text="Winning model",
        title="Best Model per Task (metrics are task-specific; do not compare heights across tasks)"
    )
    fig.show()


,Task,Winning model,Selection metric,Metric value
0,Regression,Random Forest Regressor,Lowest RMSE,1395.6848
1,Classification,Random Forest Classifier,Highest Macro-F1,0.9977


## 7. Decision, recommendations and conclusion
Use the executed output to complete these evidence-based statements:
1. **Data quality:** document duplicates, year anomalies, label cleanup and donor-rate outliers; distinguish flagged values from confirmed errors.
2. **Supply:** identify continents/countries with lower donor rates and organs that account for the largest share of transplants.
3. **Demand:** prioritize investigation of country-organ records with high waiting-list-to-transplant ratios, checking denominators and data coverage.
4. **Outcomes:** describe organ-specific survival differences and avoid causal claims from volume correlations.
5. **Statistical evidence:** report H₀/H₁, test statistic, p-value, α and effect size where applicable.
6. **Model choice:** report the best regression, classification and clustering models separately, based on held-out metrics.
7. **Limitations:** verify source provenance, year coverage, units, duplicate key meaning, synthetic/estimated values and target leakage. `Avg_Wait_Years_US` is US-labelled and should not be represented as every country's waiting time.


## Additional standalone cell — Hypothesis Testing
This supplementary test does not modify or replace any of the original questions. It tests whether the distribution of five-year survival rates differs between the two `Can_Living_Donate` groups.

In [23]:
# Standalone hypothesis test (supplementary; original questions remain unchanged)
# H0: Five-year survival rates have the same distribution in the two living-donation groups.
# H1: Five-year survival-rate distributions differ between the two groups.
# Significance level: alpha = 0.05

from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

alpha = 0.05
required_cols = ["Can_Living_Donate", "Survival_Rate_5Y_Pct"]

if not all(col in df.columns for col in required_cols):
    print("Hypothesis test could not run. Required columns:", required_cols)
else:
    ht = df[required_cols].copy()
    ht["Can_Living_Donate"] = ht["Can_Living_Donate"].astype("string").str.strip()
    ht["Survival_Rate_5Y_Pct"] = pd.to_numeric(ht["Survival_Rate_5Y_Pct"], errors="coerce")
    ht = ht.dropna(subset=required_cols)
    ht = ht[ht["Can_Living_Donate"] != ""]
    groups = list(ht["Can_Living_Donate"].unique())

    print("H0: Five-year survival-rate distributions are equal across the two groups.")
    print("H1: Five-year survival-rate distributions differ across the two groups.")
    print(f"Significance level (alpha): {alpha}")
    print(f"Usable observations: {len(ht)}")

    if len(groups) != 2:
        print(f"Test not run: expected exactly 2 groups, found {len(groups)}: {groups}")
    else:
        g1 = ht.loc[ht["Can_Living_Donate"] == groups[0], "Survival_Rate_5Y_Pct"]
        g2 = ht.loc[ht["Can_Living_Donate"] == groups[1], "Survival_Rate_5Y_Pct"]
        if len(g1) == 0 or len(g2) == 0:
            print("Test not run because one group has no usable observations.")
        else:
            u_stat, p_value = stats.mannwhitneyu(g1, g2, alternative="two-sided")
            # Rank-biserial effect size; sign depends on displayed group order.
            effect_size = (2 * u_stat / (len(g1) * len(g2))) - 1
            print(f"Group {groups[0]}: n={len(g1)}, median={g1.median():.3f}")
            print(f"Group {groups[1]}: n={len(g2)}, median={g2.median():.3f}")
            print(f"Mann–Whitney U statistic: {u_stat:.3f}")
            print(f"p-value: {p_value:.6g}")
            print(f"Rank-biserial effect size: {effect_size:.3f}")
            if p_value < alpha:
                print("Decision: Reject H0. There is statistical evidence of a difference in the survival-rate distributions.")
            else:
                print("Decision: Fail to reject H0. There is insufficient statistical evidence of a difference in the survival-rate distributions.")
            print("Interpretation: This is an association in the dataset, not evidence that living-donation eligibility causes survival differences.")

            plot_data = ht.copy()
            plot_data.boxplot(column="Survival_Rate_5Y_Pct", by="Can_Living_Donate", grid=False)
            plt.title("Five-Year Survival Rate by Living-Donation Group")
            plt.suptitle("")
            plt.xlabel("Can_Living_Donate")
            plt.ylabel("Five-Year Survival Rate (%)")
            plt.tight_layout()
            plt.show()

H0: Five-year survival-rate distributions are equal across the two groups.
H1: Five-year survival-rate distributions differ across the two groups.
Significance level (alpha): 0.05
Usable observations: 2058
Test not run: expected exactly 2 groups, found 6: ['No', 'Rare (partial)', 'Rare (lobar)', 'Rare', 'Yes', 'Yes (partial)']
